# GPT 시리즈: 언어모델의 스케일링 - 실습 코드 3: GPT 스타일 언어모델 학습 루프 (PyTorch)

- Tutorial ID: `expand-gpt-series`
- Tutorial: GPT 시리즈: 언어모델의 스케일링
- Section ID: `expand-gpt-series-code-3`
- Section: 실습 코드 3: GPT 스타일 언어모델 학습 루프 (PyTorch)

---

> 📘 **안내**: 이 노트북은 원본 실습 코드에 처음 공부하는 분들을 위한 상세한 설명과 단계별 예제를 추가한 버전입니다. 새로운 개념이 나올 때마다 (1) 아주 작은 장난감 예제로 먼저 감을 잡고 → (2) 실제 모델 코드에 적용해보는 순서로 구성했습니다. 마크다운 설명을 먼저 읽고, 코드 셀을 순서대로 실행하면서 출력되는 shape과 값을 직접 눈으로 확인해보세요.

## 이 실습에서 배우는 것

1. **Causal mask**: "미래 토큰을 못 보게 막는다"는 말이 attention score/softmax 레벨에서 실제로 어떻게 구현되는지
2. **PyTorch로 GPT 조립하기**: `nn.TransformerDecoderLayer`로 decoder-only 구조를 만드는 방법과, 그 과정에서 아주 흔히 놓치는 함정 하나
3. **생성(generate) 전략**: temperature, top-k, top-p(nucleus) 샘플링이 다음 토큰 후보의 확률분포를 각각 어떻게 바꾸는지
4. **학습 루프**: Next-token Prediction, cross-entropy loss, gradient clipping, AdamW/weight decay, cosine annealing scheduler, perplexity
5. 아주 작은 장난감(toy) 데이터셋으로 학습 루프를 **처음부터 끝까지 실제로 돌려서** loss가 줄어드는 과정과 생성 품질이 좋아지는 과정을 직접 눈으로 확인

## 미리 알고 있으면 좋은 것

- Python 기초 문법 (함수, 반복문, 클래스)
- PyTorch tensor의 기본 개념 (shape, 인덱싱, `.view()` 등)
- (선택) attention의 기본 아이디어 — 몰라도 괜찮습니다. 이번 노트북에 필요한 부분은 아래에서 간단히 다시 짚고 넘어갑니다.

## 실행 환경

- 이 노트북은 PyTorch에 의존합니다. Google Colab, 로컬 Jupyter, 또는 서버 노트북 환경에서 실행하는 것을 권장합니다.
- GPU가 없어도 괜찮습니다. 이 노트북의 모든 예제는 CPU에서도 몇 초~몇 분 안에 끝나도록 작게 설계했습니다.

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 3: GPT 스타일 언어모델 학습 루프 (PyTorch)
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, top_k, top_p 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 이번 노트북에서 조금 더 다루는 것:
#   - PyTorch 기본 클래스(nn.TransformerDecoderLayer)로 decoder-only 구조를 만들 때
#     흔히 놓치는 함정(causal mask가 절반만 적용되는 문제)을 실험으로 직접 확인합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - torch 의존 코드이므로 Colab/로컬/서버 노트북 실행을 권장합니다.
# ============================================================

## 0. 전체 그림 먼저 보기

본격적으로 코드를 보기 전에, 이 노트북에서 만들 모델이 전체적으로 어떤 흐름으로 동작하는지 그림으로 먼저 훑어봅니다. 아래 각 단계는 이후 섹션에서 하나씩 코드로 확인합니다.

```
토큰 ID (정수)                      예) [3, 17, 892, 5]
     │
     ▼  ① 토큰 임베딩 + 위치 임베딩
[벡터] = tok_emb(토큰) + pos_emb(위치)
     │
     ▼  ② Transformer 블록 × n_layers 반복
     │     (그 안에서: causal self-attention → FFN)
     │     "미래 토큰은 못 본다"는 규칙이 여기서 적용됨
     ▼
[벡터]
     │
     ▼  ③ 마지막 LayerNorm
     ▼  ④ 출력 projection (weight tying으로 임베딩과 가중치 공유)
[각 단어에 대한 점수 = logit]   shape: (배치, 시퀀스길이, vocab_size)
     │
     ▼  ⑤ (학습 시) cross-entropy loss로 "다음 토큰 맞히기" 훈련
     ▼  (생성 시) temperature/top-k/top-p로 확률분포를 조절해 다음 토큰 샘플링
```

이제 ①~⑤를 실제 코드로 하나씩 따라가 봅니다. 순서는 다음과 같습니다.

1. 라이브러리 불러오기
2. (준비운동) causal mask가 실제로 미래를 가리는 모습 확인
3. (준비운동) temperature / top-k / top-p가 확률분포를 바꾸는 모습 확인
4. `SimpleGPT` 모델 클래스 설계 및 구현
5. 모델을 만들어서 shape 확인 + "정말로 causal한지" 실험으로 검증
6. 학습 전 모델로 생성해보기 (아직은 엉망)
7. 학습 루프(`train_gpt`) 설계
8. 장난감 데이터셋 준비
9. 실제로 학습시키고, 학습 전/후 결과를 비교

## 1. 라이브러리 불러오기

- `torch`: 텐서 연산과 자동미분(autograd)을 제공하는 핵심 라이브러리
- `torch.nn`: 신경망 레이어(임베딩, Linear, TransformerDecoderLayer 등)를 담고 있는 모듈
- `torch.nn.functional` (관례적으로 `F`): softmax, cross_entropy처럼 '학습되는 파라미터가 없는' 함수형 연산 모음
- `torch.utils.data`: 미니배치를 만들어주는 `TensorDataset`, `DataLoader`
- `matplotlib.pyplot`: 학습 중 loss가 줄어드는 과정을 그래프로 보기 위해 사용

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# 실행할 때마다 결과가 달라지면 원인 파악이 어려우므로,
# 재현 가능한(reproducible) 결과를 위해 랜덤 시드를 고정합니다.
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA(GPU) 사용 가능 여부: {torch.cuda.is_available()}")

## 2. (준비운동) Causal Mask: "미래를 못 보게 막는다"는 게 무슨 뜻일까?

GPT는 **왼쪽에서 오른쪽으로** 문장을 읽으며 다음 단어를 예측합니다. 그런데 학습할 때는 정답 문장 전체를 한 번에 모델에 넣기 때문에, 아무 장치가 없다면 모델이 "다음 단어"를 미리 슬쩍 봐버리는 반칙을 저지를 수 있습니다 (시험지의 답안을 미리 커닝하는 것과 비슷합니다).

이걸 막는 장치가 **causal mask(인과적 마스크)**입니다. 아이디어는 간단합니다.

- attention score(각 토큰이 다른 토큰을 얼마나 참고할지 나타내는 점수) 행렬에서, "미래 위치"에 해당하는 칸에 `-inf`(음의 무한대)를 더해버립니다.
- 그 상태로 softmax를 취하면, `-inf`였던 칸은 확률이 정확히 `0`이 되어 사실상 "참고하지 않음"이 됩니다.

먼저 아주 작은 예제로 이 원리를 직접 눈으로 확인해봅니다.

In [ ]:
# 문장 길이가 5인 아주 짧은 예시를 가정합니다 (예: "나는 오늘 학교에 가서 잔다" 같은 5토큰 문장)
T_demo = 5

# nn.Transformer.generate_square_subsequent_mask(T)는 (T, T) 크기의 마스크를 만들어줍니다.
# - 행(row) = "지금 이 토큰 입장에서"
# - 열(col) = "이 위치를 볼 수 있는가?"
# 자기 자신과 그 이전(왼쪽)은 0(허용), 미래(오른쪽)는 -inf(차단)
causal_mask = nn.Transformer.generate_square_subsequent_mask(T_demo)
print("Causal Mask (T=5):")
print(causal_mask)
print("→ 0행(첫 토큰)은 자기 자신만 보이고, 4행(마지막 토큰)은 전부 보입니다.")

In [ ]:
# 실제로 이 마스크가 attention score에 어떤 영향을 주는지 확인해봅니다.
# (아래 점수는 실제 Q·K 연산 결과가 아니라, 원리를 보여주기 위한 무작위 값입니다)
fake_attention_scores = torch.randn(T_demo, T_demo)
print("마스크 적용 전 점수 (무작위 예시):")
print(fake_attention_scores)

masked_scores = fake_attention_scores + causal_mask
print("\n마스크를 더한 후 점수 (미래 위치는 -inf로 바뀜):")
print(masked_scores)

attention_weights = F.softmax(masked_scores, dim=-1)
print("\nsoftmax 적용 후 최종 attention 가중치 (미래 위치는 정확히 0.0):")
print(attention_weights)

print("\n각 행의 합 (확률이므로 항상 1.0이어야 정상):")
print(attention_weights.sum(dim=-1))

방금 확인한 "마스크 더하기 → softmax" 원리는 실제 PyTorch의 attention 모듈 안에서도 똑같이 일어납니다. 이번엔 진짜 `nn.MultiheadAttention`에 causal mask를 넣어서, Q/K/V가 다뤄지는 shape과 함께 확인해봅니다.

`nn.MultiheadAttention`은 입력으로 들어온 `(B, T, d_model)` 벡터를 내부에서 head 개수만큼 쪼갭니다. 즉 `head_dim = d_model / num_heads` 입니다. 각 head가 독립적으로 자기만의 Q, K, V를 만들어 attention을 계산한 뒤, 그 결과를 다시 이어붙여(concat) 원래 차원으로 되돌립니다. 그래서 입력과 출력의 shape은 `(B, T, d_model)`로 동일합니다.

In [ ]:
d_model_demo = 8
n_heads_demo = 2
T_demo2 = 4
B_demo = 1

mha = nn.MultiheadAttention(embed_dim=d_model_demo, num_heads=n_heads_demo, batch_first=True)
x_demo = torch.randn(B_demo, T_demo2, d_model_demo)
causal_mask_demo = nn.Transformer.generate_square_subsequent_mask(T_demo2)

# self-attention이므로 query, key, value 자리에 모두 같은 x_demo를 넣습니다.
attn_output, attn_weights = mha(
    x_demo, x_demo, x_demo,
    attn_mask=causal_mask_demo,
    need_weights=True,
    average_attn_weights=False,  # head별 가중치를 각각 보고 싶으므로 평균내지 않음
)

print(f"입력 x shape         : {x_demo.shape}   (B={B_demo}, T={T_demo2}, d_model={d_model_demo})")
print(f"attention 출력 shape  : {attn_output.shape}   (입력과 동일한 B, T, d_model)")
print(f"attention weight shape: {attn_weights.shape}   (B, num_heads, T, T)")
print(f"head_dim = d_model / num_heads = {d_model_demo} / {n_heads_demo} = {d_model_demo // n_heads_demo}")

print("\n첫 번째 head의 attention weight (행=현재 위치, 열=참고하는 위치):")
print(attn_weights[0, 0])
print("→ 오른쪽 위 삼각형(미래 위치)이 모두 0인 것을 확인하세요. 이게 causal mask의 효과입니다.")

## 3. (준비운동) Temperature, Top-k, Top-p: "다음 토큰을 어떻게 고를까?"

모델이 각 단어 후보에 대해 점수(logit)를 뽑아내면, 이걸 그대로 쓰는 게 아니라 **어떤 전략으로 다음 토큰을 샘플링할지**를 정해야 합니다. 세 가지 대표적인 도구를 하나씩 감을 잡아봅니다.

### 3-1. Temperature (온도)

logit을 그대로 softmax에 넣지 않고, `logit / temperature`를 넣습니다.

- `temperature < 1` → 분포가 더 **뾰족해짐** (가장 확률 높은 토큰을 더 확신 있게 선택 → 안정적이지만 단조로운 출력)
- `temperature = 1` → 원래 분포 그대로
- `temperature > 1` → 분포가 더 **완만해짐** (여러 후보가 비슷한 확률을 가지게 됨 → 다양하지만 엉뚱해질 위험이 있는 출력)

### 3-2. Top-k

확률이 높은 순서로 **상위 k개만** 후보로 남기고, 나머지는 전부 후보에서 제외(`-inf`)합니다. 후보 개수가 항상 정확히 k개로 고정됩니다.

### 3-3. Top-p (nucleus sampling)

확률이 높은 순서대로 후보를 하나씩 더해가다가, **누적 확률이 p(예: 0.9)를 넘는 순간까지의 후보만** 남깁니다. Top-k와 달리 후보 개수가 상황에 따라 달라집니다.

- 모델이 아주 확신에 차 있다면(한두 개 토큰에 확률이 몰려있다면) → 후보를 1~2개로 좁힘
- 모델이 여러 선택지를 놓고 고민 중이라면(확률이 고르게 퍼져 있다면) → 훨씬 많은 후보를 남김

즉 top-p는 상황에 따라 후보 수를 **스스로 조절**한다는 점이 top-k와 다릅니다. 아래에서 하나씩 코드로 확인합니다.

In [ ]:
# 5개의 단어 후보에 대한 점수(logit)가 있다고 가정합니다.
logits_example = torch.tensor([2.0, 1.0, 0.1, 3.0, 0.5])

for temp in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits_example / temp, dim=-1)
    print(f"temperature={temp}: {probs}")
    print(f"   -> 가장 높은 확률: {probs.max().item():.4f} (1위 후보에 대한 확신 정도)")

In [ ]:
logits_example2 = torch.tensor([2.0, 1.0, 0.1, 3.0, 0.5, 4.0, -1.0])
k = 3

# 상위 k개의 값과 인덱스를 찾습니다.
values, indices = torch.topk(logits_example2, k)
print(f"원래 logits    : {logits_example2}")
print(f"상위 {k}개 값   : {values}")
print(f"상위 {k}개 인덱스: {indices}")

# k번째로 큰 값(values의 마지막 원소)보다 작은 logit은 모두 -inf로 만들어 후보에서 제외
filtered_logits = logits_example2.clone()
threshold = values[-1]
filtered_logits[filtered_logits < threshold] = float('-inf')
print(f"\n필터링 후 logits: {filtered_logits}")

probs = F.softmax(filtered_logits, dim=-1)
print(f"최종 확률분포 (하위 후보는 정확히 0): {probs}")

In [ ]:
logits_example3 = torch.tensor([2.0, 1.0, 0.1, 3.0, 0.5, 4.0, -1.0])
p = 0.9

probs3 = F.softmax(logits_example3, dim=-1)
sorted_probs, sorted_indices = torch.sort(probs3, descending=True)
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

print(f"정렬된 확률(큰 순): {sorted_probs}")
print(f"정렬된 인덱스     : {sorted_indices}")
print(f"누적 확률         : {cumulative_probs}")

# 누적 확률이 p를 "처음 넘는" 위치까지만 후보로 남깁니다.
cutoff = (cumulative_probs >= p).nonzero()[0].item()
keep_indices = sorted_indices[:cutoff + 1]
print(f"\n누적확률이 {p}를 처음 넘는 위치: {cutoff} (0-indexed)")
print(f"최종적으로 남기는 토큰 인덱스: {keep_indices.tolist()}  (총 {len(keep_indices)}개)")

print("\n--- 비교: 분포가 뾰족할 때 vs 완만할 때, top-p가 남기는 후보 수가 달라짐 ---")
sharp_logits = torch.tensor([10.0, 0.1, 0.1, 0.1, 0.1])    # 한 후보에 확신이 몰린 경우
flat_logits = torch.tensor([1.0, 1.05, 0.95, 1.02, 0.98])  # 후보들이 고만고만한 경우

for name, lg in [("뾰족한 분포 (확신)", sharp_logits), ("완만한 분포 (불확실)", flat_logits)]:
    pr = F.softmax(lg, dim=-1)
    sp, _ = torch.sort(pr, descending=True)
    cp = torch.cumsum(sp, dim=-1)
    cutoff_ = (cp >= 0.9).nonzero()[0].item()
    print(f"{name}: top-p=0.9 일 때 남는 후보 수 = {cutoff_ + 1}개")

## 4. `SimpleGPT` 설계도

이제 준비운동에서 확인한 개념들을 이용해 실제 GPT 스타일 모델을 만들어봅니다. 코드를 보기 전에 구조를 먼저 말로 정리합니다.

**모델 구성 요소**

| 구성 요소 | 역할 |
|---|---|
| `tok_emb` | 토큰 ID(정수) → `d_model` 차원 벡터로 변환하는 조회 테이블 |
| `pos_emb` | "몇 번째 위치인지"를 알려주는 벡터 (GPT-2처럼 학습되는 위치 임베딩 사용, 최대 1024 위치) |
| `blocks` | `nn.TransformerDecoderLayer`를 `n_layers`개 쌓은 것. 각 블록 안에서 causal self-attention + FFN이 일어남 |
| `ln_f` | 마지막에 한 번 더 값의 분포를 정리하는 LayerNorm |
| `head` | `d_model` 차원 벡터 → `vocab_size` 차원 점수(logit)로 변환하는 출력층 |

**⚠️ 함정 주의: `nn.TransformerDecoderLayer`를 GPT처럼 쓰려면**

`nn.TransformerDecoderLayer`는 원래 번역 모델처럼 **인코더-디코더 구조**를 위해 설계되었습니다. forward에 두 가지 입력을 받습니다.

- `tgt`: 디코더가 지금 처리 중인 시퀀스
- `memory`: 인코더가 만들어낸 "참고할 원문 정보"

내부적으로 **self-attention(tgt끼리) → cross-attention(tgt가 memory를 참고) → FFN** 순서로 진행됩니다. GPT는 인코더가 없는 decoder-only 구조이므로, 이 노트북에서는 `memory` 자리에 `tgt`와 똑같은 `x`를 넣어서 cross-attention이 사실상 self-attention처럼 동작하도록 우회해서 사용합니다. (실무에서는 이런 우회 대신 `nn.TransformerEncoderLayer` + causal mask를 쓰는 경우가 더 많지만, 여기서는 PyTorch 기본 제공 클래스만으로 GPT 구조를 빠르게 조립해보는 연습을 합니다.)

**그런데 여기서 아주 흔히 놓치는 부분이 하나 있습니다.** self-attention에는 `tgt_mask`(causal mask)를 넣어 미래를 가리더라도, cross-attention에 적용되는 `memory_mask`를 깜빡하고 안 넣으면 어떻게 될까요? memory에는 미래 토큰의 정보가 그대로 들어있고, cross-attention은 마스킹 없이 memory의 아무 위치나 자유롭게 참고할 수 있으므로 **미래 정보가 그대로 새어 들어옵니다.** 즉 절반만 막고 절반은 뚫려있는 상태가 됩니다. 실제로 얼마나 새는지는 모델을 다 만든 뒤 바로 이어지는 5-1 섹션에서 실험으로 직접 확인해볼 것입니다. 그래서 아래 코드에서는 `tgt_mask`와 `memory_mask`에 **동일한 causal mask를 둘 다** 넣어줍니다.

**Weight Tying (가중치 공유)**

입력 토큰 임베딩(`tok_emb.weight`, shape `(vocab_size, d_model)`)과 출력 projection(`head.weight`, shape도 동일하게 `(vocab_size, d_model)`)은 마침 shape이 같습니다. GPT-2는 이 두 행렬을 아예 같은 파라미터로 묶어버리는데("weight tying"), 이렇게 하면 파라미터 수가 줄어들고 입력/출력 단어 표현이 일관되게 학습되는 효과가 있습니다.

In [ ]:
class SimpleGPT(nn.Module):
    """
    GPT-2 스타일 언어모델 (간소화 버전)

    구조 요약:
      토큰 임베딩 + 위치 임베딩
        -> Transformer 블록 x n_layers  (causal self-attention + FFN)
        -> LayerNorm
        -> 출력 projection (weight tying으로 vocab_size 크기의 logit 생성)
    """

    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=12):
        super().__init__()

        # 1) 토큰 임베딩: 정수 토큰 ID -> d_model 차원 벡터로 바꿔주는 '조회 테이블'
        #    weight shape: (vocab_size, d_model)
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # 2) 위치 임베딩: "몇 번째 토큰인지"를 알려주는 벡터
        #    GPT-2와 동일하게 최대 1024개 위치까지 지원 (학습되는 임베딩 사용)
        self.pos_emb = nn.Embedding(1024, d_model)

        # 3) Transformer 블록을 n_layers개 쌓습니다.
        self.blocks = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_model * 4,  # FFN 중간 차원은 보통 d_model의 4배 (GPT-2 관례)
                dropout=0.1,
                activation='gelu',            # GPT 계열은 ReLU 대신 GELU를 주로 사용
                batch_first=True,             # 입력 shape을 (batch, seq, d_model) 순서로 사용
            ) for _ in range(n_layers)
        ])

        # 4) 마지막 LayerNorm: 출력 직전에 값의 분포를 한 번 더 정리 (GPT-2 구조 특징)
        self.ln_f = nn.LayerNorm(d_model)

        # 5) 출력 projection: d_model 차원 벡터 -> vocab_size 차원 점수(logit)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # 6) Weight Tying (가중치 공유, GPT-2 핵심 기법)
        #    두 행렬 모두 shape이 (vocab_size, d_model)로 동일하므로 공유가 가능합니다.
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        """
        idx: (B, T) 크기의 정수 텐서. 각 값은 토큰 ID (0 ~ vocab_size-1)
        반환값 logits: (B, T, vocab_size) - 각 위치에서 "다음 토큰"에 대한 점수
        """
        B, T = idx.shape  # B: 배치 크기, T: 시퀀스 길이(토큰 개수)

        # 각 위치를 0, 1, 2, ..., T-1 로 표현 (예: T=5 -> [0,1,2,3,4])
        pos = torch.arange(T, device=idx.device)

        # 토큰 임베딩 (B, T, d_model) + 위치 임베딩 (T, d_model)
        # -> 브로드캐스팅으로 배치 전체에 위치 정보가 동일하게 더해져서 (B, T, d_model)
        x = self.tok_emb(idx) + self.pos_emb(pos)

        # Causal mask 생성: (T, T) 크기, 미래 위치는 -inf, 현재/과거 위치는 0
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)

        # Transformer 블록을 순서대로 통과
        # memory=x 로 넣어 decoder-only 구조를 흉내내되,
        # tgt_mask와 memory_mask에 모두 causal_mask를 넣어야 미래 정보가 새지 않습니다.
        # (memory_mask를 빼먹으면 어떻게 되는지는 바로 다음 5-1 섹션의 실험에서 확인합니다)
        for block in self.blocks:
            x = block(x, memory=x, tgt_mask=causal_mask, memory_mask=causal_mask)

        x = self.ln_f(x)          # (B, T, d_model)
        logits = self.head(x)     # (B, T, vocab_size)
        return logits

    @torch.no_grad()  # 생성(추론) 단계에서는 gradient 계산이 필요 없으므로 메모리/속도 절약
    def generate(self, idx, max_new_tokens=100, temperature=1.0, top_k=50, top_p=None):
        """
        자기회귀(autoregressive) 텍스트 생성.
        idx: (B, T) 시작 토큰 시퀀스 (예: 프롬프트)
        토큰을 하나씩 예측해서 idx 뒤에 이어 붙이는 과정을 max_new_tokens번 반복합니다.

        주의: top_k는 vocab_size보다 클 수 없습니다.
              (vocab_size=4인데 top_k=50이면 "RuntimeError: selected index k out of range" 발생)
        """
        for _ in range(max_new_tokens):
            # 위치 임베딩이 최대 1024개까지만 있으므로, 최근 256 토큰만 사용해 안전하게 자릅니다.
            idx_cond = idx[:, -256:]

            # 모델에 넣어 logits을 얻고, 그중 "마지막 위치"의 예측만 사용합니다.
            # (마지막 위치의 logits이 바로 '다음에 올 토큰'에 대한 예측이기 때문)
            logits = self(idx_cond)[:, -1, :] / temperature  # temperature로 분포의 뾰족함 조절

            # Top-K 필터링: 상위 top_k개만 남기고 나머지는 후보에서 제외
            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            # Top-p(nucleus) 필터링: 누적 확률이 top_p를 넘는 지점까지만 후보로 남김
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                # 누적확률이 top_p를 "처음 넘는 지점" 다음부터 제거 대상으로 표시
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
                sorted_indices_to_remove[:, 0] = False  # 1등 후보는 항상 유지 (전부 제거되는 극단적 상황 방지)

                for b in range(logits.size(0)):
                    remove_idx = sorted_indices[b][sorted_indices_to_remove[b]]
                    logits[b, remove_idx] = float('-inf')

            probs = F.softmax(logits, dim=-1)                     # logit -> 확률분포
            next_token = torch.multinomial(probs, num_samples=1)  # 확률에 따라 토큰 하나를 무작위로 선택
            idx = torch.cat([idx, next_token], dim=1)             # 새로 뽑은 토큰을 시퀀스 뒤에 이어붙임
        return idx

## 5. 모델을 실제로 만들어서 shape 확인하기

GPT-2가 사용하는 토크나이저의 vocab 크기(50257)는 그대로 사용하되, 나머지 하이퍼파라미터(`d_model`, `n_heads`, `n_layers`)는 개인 컴퓨터에서도 빠르게 실습할 수 있도록 작게 설정합니다. (참고: 실제 GPT-2 Small은 `d_model=768`, `n_heads=12`, `n_layers=12`입니다.)

In [ ]:
torch.manual_seed(42)

model = SimpleGPT(vocab_size=50257, d_model=256, n_heads=4, n_layers=4)

params = sum(p.numel() for p in model.parameters())
print(f"모델 파라미터 개수: {params:,}")

# 배치 크기 2, 시퀀스 길이 64짜리 '가짜' 입력으로 forward pass가 잘 도는지 확인
x = torch.randint(0, 50257, (2, 64))
out = model(x)
print(f"입력 shape: {x.shape} -> 출력 shape: {out.shape}")
print("  (문장 2개, 각 64개 토큰, 각 위치마다 50257개 단어 각각에 대한 점수)")

# weight tying이 실제로 걸려있는지 확인 (같은 파라미터 객체를 가리키는지)
print(f"\nweight tying 확인 (tok_emb.weight is head.weight): {model.tok_emb.weight is model.head.weight}")
print(f"tok_emb.weight shape: {model.tok_emb.weight.shape}")
print(f"head.weight shape   : {model.head.weight.shape}")

### 5-1. 실험: 우리 모델이 정말로 "미래를 못 보는지" 확인해보기

앞서 언급한 함정(4번 섹션)을 실제로 확인해봅니다. 방법은 간단합니다.

1. 아무 입력이나 하나 만들고, 모델에 통과시켜 **앞쪽 위치**의 출력값을 기록합니다.
2. 그 입력의 **맨 마지막(미래) 위치의 값만** 완전히 다른 값으로 바꿔치기 합니다.
3. 다시 모델에 통과시켜 같은 **앞쪽 위치**의 출력값을 봅니다.

모델이 정말 causal하다면, 미래 위치를 아무리 바꿔도 그보다 앞선 위치의 출력은 **한 치도 변하면 안 됩니다** (애초에 참고하지 않았어야 하니까요). `memory_mask`를 빼먹은 "버그 버전"과, 제대로 넣은 "수정 버전"을 나란히 비교합니다. (아래 두 버전은 공정한 비교를 위해 완전히 동일한 초기 가중치에서 시작합니다)

In [ ]:
torch.manual_seed(0)
d_model_t, n_heads_t, T_t, n_layers_t = 16, 4, 6, 3

# 비교를 위해 완전히 동일한 초기 가중치를 가진 블록 두 세트를 준비합니다.
blocks_buggy = nn.ModuleList([
    nn.TransformerDecoderLayer(d_model=d_model_t, nhead=n_heads_t, dim_feedforward=32,
                                dropout=0.0, batch_first=True)
    for _ in range(n_layers_t)
])
blocks_fixed = nn.ModuleList([
    nn.TransformerDecoderLayer(d_model=d_model_t, nhead=n_heads_t, dim_feedforward=32,
                                dropout=0.0, batch_first=True)
    for _ in range(n_layers_t)
])
blocks_fixed.load_state_dict(blocks_buggy.state_dict())  # 두 세트의 가중치를 동일하게 맞춤
for b in list(blocks_buggy) + list(blocks_fixed):
    b.eval()  # dropout 끄기 (dropout=0.0이라 큰 의미는 없지만 습관적으로 명시)

mask_t = nn.Transformer.generate_square_subsequent_mask(T_t)
x_orig = torch.randn(1, T_t, d_model_t)
x_changed = x_orig.clone()
x_changed[0, T_t - 1, :] += 1000.0  # 맨 마지막(미래) 토큰만 완전히 다른 값으로 바꿔치기


def run_stack(blocks, use_memory_mask, inp):
    h = inp
    for blk in blocks:
        if use_memory_mask:
            h = blk(h, memory=h, tgt_mask=mask_t, memory_mask=mask_t)
        else:
            h = blk(h, memory=h, tgt_mask=mask_t)  # memory_mask 누락!
    return h


with torch.no_grad():
    out_buggy_orig = run_stack(blocks_buggy, use_memory_mask=False, inp=x_orig)
    out_buggy_changed = run_stack(blocks_buggy, use_memory_mask=False, inp=x_changed)
    diff_buggy = (out_buggy_orig[0, 0, :] - out_buggy_changed[0, 0, :]).abs().max().item()

    out_fixed_orig = run_stack(blocks_fixed, use_memory_mask=True, inp=x_orig)
    out_fixed_changed = run_stack(blocks_fixed, use_memory_mask=True, inp=x_changed)
    diff_fixed = (out_fixed_orig[0, 0, :] - out_fixed_changed[0, 0, :]).abs().max().item()

print(f"{n_layers_t}개 레이어를 쌓았을 때, '위치 0'의 출력이 '맨 마지막(미래) 토큰' 변경에 반응하는 정도\n")
print(f"[버그 버전] memory_mask 없음       : {diff_buggy:.6f}   <- 0이 아님 = 미래 정보가 샜다는 뜻!")
print(f"[수정 버전] memory_mask=causal_mask: {diff_fixed:.10f}   <- 정확히 0 = 진짜 causal")

실행해보면 버그 버전은 0이 아닌 값이, 수정 버전은 정확히 0.0이 나오는 것을 확인할 수 있습니다. 이것이 바로 우리 `SimpleGPT.forward()`에서 `tgt_mask`뿐 아니라 `memory_mask`까지 causal mask로 채워 넣은 이유입니다.

> 💡 **왜 이게 중요할까요?** 이 구멍을 막지 않고 학습시키면, 모델은 진짜 언어 패턴을 배우는 대신 "정답이 이미 입력 어딘가에 들어있으니 그냥 그걸 그대로 베끼면 되는" **지름길(shortcut)**을 배워버릴 수 있습니다. 이런 모델은 학습 중 loss는 아주 낮게(좋아 보이게) 나오지만, 정작 실제 생성(`generate()`) 시점에는 미래 토큰이라는 게 애초에 존재하지 않기 때문에 그 지름길을 쓸 수 없어 형편없는 결과를 내는 **학습-추론 불일치(train/inference mismatch)** 문제가 생깁니다. "loss가 잘 떨어진다고 안심하지 말고, 정말로 의도한 규칙대로 동작하는지 직접 실험으로 확인하는 습관"이 이 섹션의 핵심 교훈입니다.

## 6. 학습 전 모델로 생성해보기

아직 한 번도 학습시키지 않았으니, 이 모델은 문장의 의미를 전혀 모릅니다. 그래도 `generate()` 함수 자체가 잘 동작하는지, 그리고 `top_k`/`top_p` 옵션을 함께 썼을 때도 에러 없이 돌아가는지 미리 확인해봅니다.

In [ ]:
model.eval()  # 생성할 때는 dropout을 끄기 위해 eval 모드로 전환

prompt = torch.randint(0, 50257, (1, 5))
print(f"prompt shape: {prompt.shape}, 내용: {prompt.tolist()}")

# top_k만 사용
out_gen = model.generate(prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f"\n[top_k=50] 생성 결과: {out_gen.tolist()}")

# top_k + top_p 함께 사용
out_gen2 = model.generate(prompt, max_new_tokens=10, temperature=0.8, top_k=50, top_p=0.9)
print(f"[top_k=50, top_p=0.9] 생성 결과: {out_gen2.tolist()}")

print("\n-> 아직 학습 전이라 의미 없는 토큰 ID가 나열될 뿐입니다. 심지어 같은 토큰이 반복될 수도 있습니다.")
print("   (임베딩이 무작위로 초기화되어 있어서 특정 토큰의 점수가 우연히 계속 높게 나올 수 있기 때문)")

## 7. 학습 루프 설계하기 — Next-token Prediction이란?

GPT의 학습 방식은 한 문장 안에서 **각 위치마다 "바로 다음 토큰이 뭘지" 맞히는** 것입니다. 정답(target)은 입력을 한 칸 왼쪽으로 민 것과 같습니다.

```
문장(예시)          :  나는    오늘    학교에    간다
입력 x              :  나는    오늘    학교에    간다
정답 (x를 1칸 shift) :         오늘    학교에    간다    (그 다음, 이 예시엔 없음)

위치 0("나는")을 보고        -> 위치 1("오늘")을 맞혀야 함
위치 0~1("나는 오늘")을 보고 -> 위치 2("학교에")를 맞혀야 함
위치 0~2 전체를 보고         -> 위치 3("간다")을 맞혀야 함
```

코드에서는 `logits[:, :-1, :]`(마지막 위치를 제외한 모든 예측)와 `x[:, 1:]`(첫 토큰을 제외한 정답)를 비교해서 이 규칙을 구현합니다.

**이 과정에서 쓰이는 도구들**

- **Cross-entropy loss**: 모델이 예측한 확률분포가 정답 토큰으로부터 얼마나 먼지를 재는 지표. (정답 토큰에 낮은 확률을 줄수록 loss가 커짐)
- **Perplexity(혼란도) = exp(loss)**: "평균적으로 몇 개의 후보 중 하나를 고르는 정도로 헷갈려하는가"에 가까운 직관적인 숫자로 바꾼 것. 1에 가까울수록 확신, 클수록 혼란스럽다는 뜻입니다.
- **Gradient Clipping**: 가끔 gradient(역전파로 계산된 변화량)가 비정상적으로 커지는 순간이 있는데, 이를 그대로 반영하면 파라미터가 널뛰듯 튀며 학습이 무너질 수 있습니다. `clip_grad_norm_`은 전체 gradient의 크기(norm)가 정해둔 값(여기서는 1.0)을 넘지 않도록 비율을 줄여주는 안전장치입니다.
- **AdamW & weight_decay**: Adam은 파라미터마다 학습 속도를 자동으로 조절해주는 옵티마이저입니다. 여기에 weight_decay(가중치 감쇠)를 더하면 파라미터 값이 지나치게 커지지 않도록 눌러주어 과적합(overfitting)을 억제하는 데 도움이 됩니다. AdamW는 이 weight decay를 더 올바른 방식으로 분리 적용한 버전으로, GPT 계열 모델의 사실상 표준 선택입니다.
- **Cosine Annealing 스케줄러**: 학습률(learning rate)을 처음엔 크게 시작해서 코사인 곡선을 그리듯 서서히 줄여나가는 방식입니다. 초반엔 크게크게 이동하며 대략적인 방향을 잡고, 후반엔 미세 조정하도록 돕습니다.

In [ ]:
def train_gpt(model, data_loader, epochs=5, lr=3e-4, log_every=100):
    """
    GPT 스타일 언어모델 학습 함수 (Next-token Prediction)
    반환값으로 매 스텝의 loss 기록을 돌려줍니다 (나중에 그래프로 그리기 위함).
    """
    # AdamW: Adam + weight decay(가중치 감쇠)를 올바르게 결합한 옵티마이저
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)

    # CosineAnnealingLR: 학습률을 코사인 곡선을 그리며 서서히 줄여나가는 스케줄러
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(data_loader))

    model.train()  # Dropout이 '학습 모드'로 동작하도록 설정 (생성 시엔 model.eval() 사용)

    loss_history = []
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (x,) in enumerate(data_loader):
            # x: (batch, seq_len) - 정수 토큰 ID로 이루어진 입력 시퀀스

            logits = model(x)  # (batch, seq_len, vocab_size)

            # -- Next-token prediction loss 계산 --
            # logits[:, :-1, :] : 마지막 위치를 제외한 모든 위치의 예측
            # x[:, 1:]          : 첫 토큰을 제외한 나머지 (=정답, 한 칸씩 밀린 시퀀스)
            #
            # cross_entropy는 입력을 (샘플 개수, 클래스 개수) 형태로 기대하므로
            # .view(-1, vocab_size) / .view(-1) 로 배치와 시퀀스 차원을 하나로 합쳐줍니다.
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                x[:, 1:].contiguous().view(-1),
            )

            optimizer.zero_grad()   # 이전 스텝에서 계산된 gradient 초기화
            loss.backward()         # 역전파로 각 파라미터의 gradient 계산

            # Gradient Clipping (GPT 학습의 핵심 안전장치 중 하나)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()   # 계산된 gradient로 파라미터 업데이트
            scheduler.step()   # 학습률(lr) 스케줄 한 스텝 진행

            total_loss += loss.item()
            loss_history.append(loss.item())

            if batch_idx % log_every == 0:
                ppl = torch.exp(loss).item()
                print(f"  Step {batch_idx}: loss={loss.item():.4f}, perplexity={ppl:.1f}")

        avg_loss = total_loss / len(data_loader)
        ppl = torch.exp(torch.tensor(avg_loss)).item()
        print(f"Epoch {epoch + 1}: avg_loss={avg_loss:.4f}, perplexity={ppl:.1f}")

    return loss_history

## 8. 학습 루프를 실제로 돌려볼 장난감(toy) 데이터셋 만들기

`train_gpt`가 실제로 loss를 줄이는지 눈으로 확인하려면 데이터가 있어야 합니다. 실제 자연어 텍스트와 GPT-2 전체 vocab(50257개)으로 학습시키면 튜토리얼 환경에서 너무 오래 걸리므로, 대신 아주 단순한 **규칙 기반 패턴**을 사용합니다.

- 토큰은 `0, 1, 2, 3` 딱 4종류뿐입니다.
- `0 → 1 → 2 → 3 → 0 → 1 → 2 → 3 → ...` 처럼 4개가 계속 반복되는 패턴입니다.
- 시작 위치(위상, phase)를 다르게 해서 여러 개의 학습 샘플을 만듭니다.

이 데이터는 실제 언어가 아니라, **학습 루프가 정상적으로 동작하고 loss가 실제로 줄어드는지**를 눈으로 확인하기 위한 최소한의 예시라는 점을 기억해주세요.

In [ ]:
torch.manual_seed(0)

TOY_VOCAB_SIZE = 4    # 토큰 종류: 0, 1, 2, 3 (4개 뿐)
SEQ_LEN = 16           # 한 시퀀스(문장)의 길이
NUM_SAMPLES = 200      # 학습 샘플 개수

pattern = [0, 1, 2, 3]         # 이 패턴이 계속 반복됩니다
long_sequence = pattern * 100  # 패턴을 100번 반복해 충분히 긴 시퀀스 생성 (총 400개)

# 시작 위치를 하나씩 옮겨가며 길이 SEQ_LEN짜리 조각을 잘라 학습 샘플을 만듭니다.
# (시작 위치가 다르므로 패턴의 '위상'이 서로 다른 샘플이 됩니다)
samples = []
for start in range(NUM_SAMPLES):
    window = long_sequence[start: start + SEQ_LEN]
    samples.append(window)

toy_data = torch.tensor(samples, dtype=torch.long)  # (NUM_SAMPLES, SEQ_LEN)
print(f"toy_data shape: {toy_data.shape}")
print(f"샘플 0 (위상 0): {toy_data[0].tolist()}")
print(f"샘플 1 (위상 1): {toy_data[1].tolist()}")
print(f"샘플 2 (위상 2): {toy_data[2].tolist()}")

# TensorDataset(toy_data)는 인덱싱할 때 (toy_data[i],) 형태의 1-튜플을 반환합니다.
# 그래서 train_gpt 안의 "for batch_idx, (x,) in enumerate(data_loader)" 와 짝이 맞습니다.
toy_dataset = TensorDataset(toy_data)
toy_loader = DataLoader(toy_dataset, batch_size=16, shuffle=True)
print(f"\n배치 크기: 16, 총 배치 개수: {len(toy_loader)}")

## 9. 학습 전 vs 학습 후 비교 실험

이제 진짜 실험을 해봅니다. 순서는 다음과 같습니다.

1. 장난감 데이터셋에 맞는 아주 작은 모델을 새로 만듭니다.
2. **학습 전** 상태에서 `"0, 1, 2"` 다음에 뭐가 이어질지 생성시켜봅니다. (정답은 `3, 0, 1, 2, 3, ...`)
3. `train_gpt`로 학습시키면서 loss가 줄어드는 것을 확인합니다.
4. loss 그래프를 그려봅니다.
5. **학습 후** 다시 같은 프롬프트로 생성시켜서, 패턴을 제대로 배웠는지 확인합니다.

In [ ]:
torch.manual_seed(0)

# 장난감 데이터에 맞춰 훨씬 작은 모델을 만듭니다 (vocab이 작아졌으므로 d_model 등도 작게)
toy_model = SimpleGPT(vocab_size=TOY_VOCAB_SIZE, d_model=32, n_heads=2, n_layers=2)

toy_params = sum(p.numel() for p in toy_model.parameters())
print(f"장난감 모델 파라미터 개수: {toy_params:,}")

toy_prompt = torch.tensor([[0, 1, 2]])  # "0, 1, 2" 다음에 뭐가 올지 예측시켜보기 (정답은 3)

toy_model.eval()
generated_before = toy_model.generate(toy_prompt, max_new_tokens=8, temperature=0.3, top_k=4)
print(f"\n[학습 전] 생성 결과: {generated_before.tolist()}")
print("-> 아직 패턴을 모르므로 뒤죽박죽이거나 같은 숫자만 반복될 가능성이 높습니다.")

In [ ]:
toy_model.train()
loss_history = train_gpt(toy_model, toy_loader, epochs=20, lr=1e-3, log_every=5)

Loss가 줄어드는 과정을 그래프로 확인해봅니다. (y축을 로그 스케일로 그려서, 처음의 급격한 하락부터 이후의 미세한 변화까지 한 그래프에서 함께 볼 수 있게 했습니다)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Loss (log scale)")
plt.title("Training Loss over Steps")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"첫 스텝 loss  : {loss_history[0]:.4f}")
print(f"마지막 스텝 loss: {loss_history[-1]:.6f}")

In [ ]:
# 학습 전과 똑같은 프롬프트([0, 1, 2])로 다시 생성시켜서 패턴을 제대로 배웠는지 확인합니다.
# temperature를 낮게 준 이유: 모델이 패턴을 확실히 배웠다면 가장 확률 높은 토큰이 항상 정답과
# 같아야 하는데, 그 "확신"이 생성 결과에도 잘 드러나도록 하기 위함입니다.
toy_model.eval()
generated_after = toy_model.generate(toy_prompt, max_new_tokens=8, temperature=0.3, top_k=4)
print(f"[학습 후] 생성 결과: {generated_after.tolist()}")
print("기대 패턴        : [[0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2]]  (0,1,2,3이 계속 반복)")

# 프롬프트 이후 부분(=모델이 새로 생성한 부분)만 따로 떼어 정답과 비교
expected_continuation = [3, 0, 1, 2, 3, 0, 1, 2]
actual_continuation = generated_after[0, 3:].tolist()
match = sum(a == e for a, e in zip(actual_continuation, expected_continuation))
print(f"\n생성된 8개 토큰 중 기대 패턴과 일치한 개수: {match} / 8")

## 정리하며

이 실습에서 다룬 내용을 정리하면 다음과 같습니다.

- **Causal mask**: attention score에 `-inf`를 더하고 softmax를 취하면 미래 위치의 확률이 정확히 0이 된다는 것을 직접 확인했습니다.
- **`nn.TransformerDecoderLayer`로 decoder-only 모델 만들기**: `memory=x`로 self-attention처럼 우회해서 쓰되, `tgt_mask`와 `memory_mask`를 모두 채워야 진짜로 causal해진다는 것을 실험으로 검증했습니다.
- **Weight Tying**: 입력 임베딩과 출력 projection이 같은 shape을 갖는다는 점을 이용해 파라미터를 공유하는 기법을 확인했습니다.
- **생성 전략**: temperature(분포의 뾰족함 조절), top-k(고정 개수 후보 제한), top-p(누적 확률 기반 가변 후보 제한)가 각각 어떻게 다음 토큰 후보를 좁히는지 비교했습니다.
- **학습 루프**: Next-token Prediction의 shift-by-one 구조, cross-entropy loss, gradient clipping, AdamW/weight decay, cosine annealing scheduler, perplexity를 하나씩 뜯어봤습니다.
- 장난감 데이터셋으로 학습 루프를 처음부터 끝까지 돌려서, loss가 실제로 줄어들고 생성 품질이 좋아지는 것을 눈으로 확인했습니다.

## 더 해보면 좋은 실험

- `d_model`, `n_heads`, `n_layers`를 바꿔가며 파라미터 수와 학습 속도가 어떻게 달라지는지 비교해보기
- 장난감 패턴을 더 복잡하게 바꿔보기 (예: 주기를 늘리거나, 두 가지 패턴을 섞기)
- `temperature`, `top_k`, `top_p` 값을 바꿔가며 `generate()`의 결과가 어떻게 달라지는지 실험해보기
- (도전 과제) 실제 텍스트 데이터셋과 진짜 토크나이저(예: GPT-2 tokenizer)로 바꿔서 학습시켜보기